In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_fscore_support, accuracy_score
)

In [8]:
# train_and_save.py  (recall-focused)
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, average_precision_score, fbeta_score
)
from joblib import dump

FEATURES_CSV = "merged_ml_dataset.csv"
MODEL_PATH   = "rf_drowsiness_model.joblib"
META_PATH    = "rf_metadata.json"

# -----------------------------
# 1) Load
# -----------------------------
df = pd.read_csv(FEATURES_CSV)

# 2) Features/labels
drop_cols = [c for c in ["label", "filename"] if c in df.columns]
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols].values
y = df["label"].values

# -----------------------------
# 3) Split: train/val/test
#    64% train / 16% val / 20% test
# -----------------------------
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, stratify=y_train_full, random_state=42
)

# -----------------------------
# 4) Model (recall-friendly defaults)
# -----------------------------
rf = RandomForestClassifier(
    n_estimators=600,        # smoother probabilities
    max_depth=14,           # allow moderately deep interactions
    min_samples_leaf=10,    # finer leaves than 20 -> better recall
    min_samples_split=20,   # regularization
    max_features="sqrt",
    class_weight="balanced",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# 5) Train
rf.fit(X_train, y_train)

# -----------------------------
# 6) Validation: choose decision threshold for recall
#    Use F2-score (beta=2 favors recall) across thresholds
# -----------------------------
val_proba = rf.predict_proba(X_val)[:, 1]

# Sweep thresholds between 0.20 and 0.80 (adjust if needed)
ths = np.linspace(0.20, 0.80, 61)
best_th, best_f2 = 0.50, -1.0
for t in ths:
    preds = (val_proba >= t).astype(int)
    f2 = fbeta_score(y_val, preds, beta=2, zero_division=0)
    if f2 > best_f2:
        best_f2, best_th = f2, float(t)

print(f"Chosen threshold on validation (by F2): {best_th:.2f}  |  F2={best_f2:.4f}")

# -----------------------------
# 7) Test set evaluation
#    (a) Default 0.50 threshold
#    (b) Tuned threshold
# -----------------------------
test_proba = rf.predict_proba(X_test)[:, 1]

# (a) Default 0.50
y_pred_50 = (test_proba >= 0.50).astype(int)
acc50 = accuracy_score(y_test, y_pred_50)
prec50, rec50, f150, _ = precision_recall_fscore_support(y_test, y_pred_50, average="binary", zero_division=0)
roc_auc = roc_auc_score(y_test, test_proba)
pr_auc  = average_precision_score(y_test, test_proba)

print("\n=== Metrics @ default threshold = 0.50 (Test) ===")
print(f"Accuracy         : {acc50:.4f}")
print(f"Precision (pos=1): {prec50:.4f}")
print(f"Recall    (pos=1): {rec50:.4f}")
print(f"F1        (pos=1): {f150:.4f}")
print(f"ROC-AUC         : {roc_auc:.4f}")
print(f"PR-AUC          : {pr_auc:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_50))
print("Classification Report:")
print(classification_report(y_test, y_pred_50, digits=4))


# -----------------------------
# 8) Top features (optional)
# -----------------------------
importances = rf.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]
print("\n=== Top 15 Feature Importances ===")
for i in top_idx:
    print(f"{feature_cols[i]:>18s} : {importances[i]:.5f}")

# -----------------------------
# 9) Save artifacts (model + feature order + tuned threshold)
# -----------------------------
dump(rf, MODEL_PATH)
with open(META_PATH, "w") as f:
    json.dump(
        {
            "model_path": MODEL_PATH,
            "feature_cols": feature_cols,   # keep order for real-time
            "threshold": best_th,          # use this in production
            "notes": "RF tuned for recall; threshold selected on validation by F2."
        },
        f, indent=2
    )

print(f"\n✅ Saved model to: {MODEL_PATH}")
print(f"✅ Saved metadata (features + threshold) to: {META_PATH}")


Chosen threshold on validation (by F2): 0.34  |  F2=0.9092

=== Metrics @ default threshold = 0.50 (Test) ===
Accuracy         : 0.8148
Precision (pos=1): 0.8418
Recall    (pos=1): 0.8144
F1        (pos=1): 0.8279
ROC-AUC         : 0.9026
PR-AUC          : 0.9140
Confusion Matrix:
[[4865 1103]
 [1337 5867]]
Classification Report:
              precision    recall  f1-score   support

           0     0.7844    0.8152    0.7995      5968
           1     0.8418    0.8144    0.8279      7204

    accuracy                         0.8148     13172
   macro avg     0.8131    0.8148    0.8137     13172
weighted avg     0.8158    0.8148    0.8150     13172


=== Top 15 Feature Importances ===
               MAR : 0.11642
           avg_EAR : 0.10224
           LBP_124 : 0.06104
           LBP_126 : 0.05642
           LBP_121 : 0.05138
             LBP_4 : 0.05034
           LBP_120 : 0.04105
           LBP_119 : 0.03710
            LBP_60 : 0.03176
             LBP_8 : 0.03095
           LBP_